In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os

import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.lines import Line2D

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION"
print(f"Base directory for data: {BASE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head(2)

In [ ]:
# --- plot KDEs ---
PLOT_COLS = MONTHLY_COLS + STATIC_COLS  # use whatever is present
COL_LABELS = {
    "t2m": "2m temperature",
    "tp": "Precipitation",
    "slhf": "Latent heat flux",
    "sshf": "Sensible heat flux",
    "ssrd": "Solar radiation",
    "fal": "Albedo",
    "str": "Thermal radiation",
    "ELEVATION_DIFFERENCE": "Elevation diff. (m)",
    "aspect": "Aspect (°)",
    "slope": "Slope (°)",
    "svf": "Sky-view factor",
}


In [ ]:
df_CH = load_stakes(cfg, "CH")
print(df_CH.GLACIER.unique())

# --- load aletsch monthly grid ---
glacier_name = 'aletsch'
rgi_id_gl = df_CH[df_CH.GLACIER == glacier_name].RGIId.values[0]

path_monthly_dfs = os.path.join(
    cfg.dataPath, 'RGI_v6/RGI_11_CentralEurope/monthly_parquets/')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_gl)

yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_gl}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_aletsch = pd.concat([pd.read_parquet(f) for f in yearly_files],
                       ignore_index=True)
df_aletsch.dropna(subset=feature_columns, inplace=True)
print(f"Shape: {df_aletsch.shape}")

df = df_aletsch[df_aletsch.YEAR == 2016]
sns.scatterplot(df, x='POINT_LON', y='POINT_LAT', hue='aspect')

In [ ]:
# # --- load aletsch monthly grid ---
# glacier_name = 'sanktanna'
# rgi_id_gl = df_CH[df_CH.GLACIER == glacier_name].RGIId.values[0]

# path_monthly_dfs = os.path.join(
#     cfg.dataPath, 'RGI_v6/RGI_11_CentralEurope/monthly_parquets/')
# path_parquet = os.path.join(path_monthly_dfs, rgi_id_gl)

# yearly_files = sorted(
#     glob.glob(os.path.join(path_parquet, f"{rgi_id_gl}_grid_*.parquet")))
# print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

# df_stanna = pd.concat([pd.read_parquet(f) for f in yearly_files],
#                       ignore_index=True)
# df_stanna.dropna(subset=feature_columns, inplace=True)
# print(f"Shape: {df_stanna.shape}")

# df = df_stanna[df_stanna.YEAR == 2016]
# sns.scatterplot(df, x='POINT_LON', y='POINT_LAT', hue='aspect')

In [ ]:
df_ALA = load_stakes(cfg, "ALA")
print(df_ALA.GLACIER.unique())

# --- load aletsch monthly grid ---
glacier_name = 'KAHILTNA'
rgi_id_gl = df_ALA[df_ALA.GLACIER == glacier_name].RGIId.values[0]
print(rgi_id_gl)
# rgi_id_gl = "RGI60-01.00013"
# glacier_name = df_ALA[df_ALA.RGIId == rgi_id_gl].GLACIER.values[0]
print(
    f"Loading monthly data for glacier {glacier_name} with RGI ID {rgi_id_gl}")

path_monthly_dfs = os.path.join(cfg.dataPath,
                                'RGI_v6/RGI_01_Alaska/monthly_parquets')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_gl)

yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_gl}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_alaska = pd.concat([pd.read_parquet(f) for f in yearly_files],
                      ignore_index=True)
df_alaska.dropna(subset=feature_columns, inplace=True)
print(f"Shape: {df_alaska.shape}")

df = df_alaska[df_alaska.YEAR == 2016]
sns.scatterplot(df, x='POINT_LON', y='POINT_LAT', hue='POINT_ELEVATION')

In [ ]:
ds = xr.open_dataset(
    cfg.dataPath + "/RGI_v6/RGI_01_Alaska/xr_masked_grids/RGI60-01.22193.zarr")
ds.masked_elev.plot()

In [ ]:
df_NOR = load_stakes(cfg, "NOR")
print(df_NOR.GLACIER.unique())

# --- load aletsch monthly grid ---
glacier_name = 'Aalfotbreen'
rgi_id_gl = df_NOR[df_NOR.GLACIER == glacier_name].RGIId.values[0]

path_monthly_dfs = os.path.join(cfg.dataPath,
                                'RGI_v6/RGI_08_Scandinavia/monthly_parquets')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_gl)

yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_gl}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_gl_nor = pd.concat([pd.read_parquet(f) for f in yearly_files],
                      ignore_index=True)
df_gl_nor.dropna(subset=feature_columns, inplace=True)
print(f"Shape: {df_gl_nor.shape}")

df = df_gl_nor[df_gl_nor.YEAR == 2016]
sns.scatterplot(df, x='POINT_LON', y='POINT_LAT', hue='aspect')

In [ ]:
df_ISL = load_stakes(cfg, "ISL")
print(df_ISL.GLACIER.unique())


In [ ]:
df_ISL = load_stakes(cfg, "ISL")
print(df_ISL.GLACIER.unique())

# --- load aletsch monthly grid ---
# glacier_name = 'Aalfotbreen'
# rgi_id_gl = df_ISL[df_ISL.GLACIER == glacier_name].RGIId.values[0]
rgi_id_gl = "RGI60-06.00340"
glacier_name = rgi_id_gl
path_monthly_dfs = os.path.join(cfg.dataPath,
                                'RGI_v6/RGI_06_Iceland/monthly_parquets')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_gl)
# make sure the directory exists
if not os.path.exists(path_parquet):
    raise FileNotFoundError(
        f"Directory for glacier {glacier_name} not found at {path_parquet}")
yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_gl}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_gl_ISL = pd.concat([pd.read_parquet(f) for f in yearly_files],
                      ignore_index=True)
df_gl_ISL.dropna(subset=feature_columns, inplace=True)
print(f"Shape: {df_gl_ISL.shape}")

df = df_gl_ISL[df_gl_ISL.YEAR == 2016]
sns.scatterplot(df, x='POINT_LON', y='POINT_LAT', hue='aspect')

In [ ]:
# --- build a dict of glacier dfs ---
GLACIERS_TO_PLOT = {
    "Aletsch (CH)": df_aletsch,
    "Aalfotbreen (NOR)": df_gl_nor,
    "Kahiltna (ALA)": df_alaska,
    # "Iceland (ISL)": df_gl_ISL,
}

COLORS = {
    "Aletsch (CH)": "#2166ac",
    "Aalfotbreen (NOR)": "#d73027",
    "Kahiltna (ALA)": "#4dac26",
    # "Iceland (ISL)": "#fdb863",
}

# selected features for the KDE panel
SELECTED_COLS = [
    "t2m", "tp", "ssrd", "sshf", "ELEVATION_DIFFERENCE", "aspect", "slope",
    "svf"
]
COL_LABELS = {
    "t2m": "2m temperature (°C)",
    "tp": "Precipitation (m/month)",
    "ELEVATION_DIFFERENCE": "Elevation diff. (m)",
    "aspect": "Aspect (°)",
    "ssrd": "Solar radiation (W/m²)",
    "svf": "Sky-view factor",
}

ncols = 4
nrows = int(np.ceil(len(SELECTED_COLS) / ncols))
fig, axes = plt.subplots(nrows,
                         ncols,
                         figsize=(4 * ncols, 3 * nrows),
                         squeeze=False)

import scipy.stats as scipy_stats

for idx, col in enumerate(SELECTED_COLS):
    ax = axes[idx // ncols][idx % ncols]

    all_vals = pd.concat(
        [df[col].dropna() for df in GLACIERS_TO_PLOT.values()])
    xmin = float(all_vals.min())  # <-- full range, no clipping
    xmax = float(all_vals.max())
    x_grid = np.linspace(xmin, xmax, 500)

    for label, df in GLACIERS_TO_PLOT.items():
        vals = df[col].dropna().values
        if len(vals) < 10:
            continue

        kde = scipy_stats.gaussian_kde(vals)  # <-- fit on all values
        y = kde(x_grid)
        y = y / y.max()

        ax.plot(x_grid, y, color=COLORS[label], linewidth=2, label=label)

    ax.set_ylim(0, 1.05)
    ax.set_title(COL_LABELS.get(col, col), fontsize=10)
    ax.set_xlabel("")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", color="#e0e0e0", linewidth=0.5)
    ax.set_axisbelow(True)

for idx in range(len(SELECTED_COLS), nrows * ncols):
    axes[idx // ncols][idx % ncols].axis("off")

handles = [
    plt.Line2D([0], [0], color=COLORS[l], linewidth=2, label=l)
    for l in GLACIERS_TO_PLOT
]
fig.legend(handles=handles,
           loc="lower center",
           ncol=len(GLACIERS_TO_PLOT),
           frameon=False,
           fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle("Feature distributions — glacier comparison", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
NOR_GLACIERS = {
    "Ålfotbreen": "Aalfotbreen",
    "Hansebreen": "Hansebreen",
    "Nigardsbreen": "Nigardsbreen",
    "Engabreen": "Engabreen",
}

df_nor_by_glacier = {}
for display_name, glacier_name in NOR_GLACIERS.items():
    try:
        rgi_id_gl = df_NOR[df_NOR.GLACIER == glacier_name].RGIId.values[0]
    except IndexError:
        print(f"Not found: {glacier_name}")
        continue

    path_parquet = os.path.join(cfg.dataPath,
                                'RGI_v6/RGI_08_Scandinavia/monthly_parquets',
                                rgi_id_gl)
    yearly_files = sorted(
        glob.glob(os.path.join(path_parquet, f"{rgi_id_gl}_grid_*.parquet")))

    if not yearly_files:
        print(f"No files found for {glacier_name}")
        continue

    df_gl = pd.concat([pd.read_parquet(f) for f in yearly_files],
                      ignore_index=True)
    df_gl.dropna(subset=feature_columns, inplace=True)
    df_nor_by_glacier[display_name] = df_gl
    print(f"Loaded {display_name}: {df_gl.shape}")

In [ ]:
ALA_GLACIERS = {
    "Gulkana": "GULKANA",
    "Lemon Creek": "LEMON CREEK",
    "Taku": "TAKU",
    "Wolverine": "WOLVERINE",
    "Kahiltna": "KAHILTNA",
}

df_ala_by_glacier = {}
for display_name, glacier_name in ALA_GLACIERS.items():
    try:
        rgi_id_gl = df_ALA[df_ALA.GLACIER == glacier_name].RGIId.values[0]
    except IndexError:
        print(f"Not found: {glacier_name}")
        continue

    path_parquet = os.path.join(cfg.dataPath,
                                'RGI_v6/RGI_01_Alaska/monthly_parquets',
                                rgi_id_gl)
    yearly_files = sorted(
        glob.glob(os.path.join(path_parquet, f"{rgi_id_gl}_grid_*.parquet")))

    if not yearly_files:
        print(f"No files found for {glacier_name}")
        continue

    df_gl = pd.concat([pd.read_parquet(f) for f in yearly_files],
                      ignore_index=True)
    df_gl.dropna(subset=feature_columns, inplace=True)
    df_ala_by_glacier[display_name] = df_gl
    print(f"Loaded {display_name}: {df_gl.shape}")

In [ ]:
# def plot_glacier_vs_aletsch_aalfot(df_ala_glacier,
#                                    ala_label,
#                                    ala_color="#4dac26"):
#     GLACIERS_TO_PLOT = {
#         "Aletsch (CH)": df_aletsch,
#         "Ålfotbreen (NOR)": df_nor_by_glacier["Ålfotbreen"],
#         ala_label: df_ala_glacier,
#     }
#     COLORS = {
#         "Aletsch (CH)": "#2166ac",
#         "Ålfotbreen (NOR)": "#d73027",
#         ala_label: ala_color,
#     }

#     ncols = 4
#     nrows = int(np.ceil(len(PLOT_COLS) / ncols))
#     fig, axes = plt.subplots(nrows,
#                              ncols,
#                              figsize=(4 * ncols, 3 * nrows),
#                              squeeze=False)

#     for idx, col in enumerate(PLOT_COLS):
#         ax = axes[idx // ncols][idx % ncols]

#         all_vals = pd.concat(
#             [df[col].dropna() for df in GLACIERS_TO_PLOT.values()])
#         xmin = float(all_vals.min())
#         xmax = float(all_vals.max())
#         x_grid = np.linspace(xmin, xmax, 500)

#         for label, df in GLACIERS_TO_PLOT.items():
#             vals = df[col].dropna().values
#             if len(vals) < 10:
#                 continue
#             kde = scipy_stats.gaussian_kde(vals)
#             y = kde(x_grid)
#             y = y / y.max()
#             ax.plot(x_grid, y, color=COLORS[label], linewidth=2, label=label)

#         ax.set_ylim(0, 1.05)
#         ax.set_title(COL_LABELS.get(col, col), fontsize=10)
#         ax.set_xlabel("")
#         ax.spines[["top", "right", "left"]].set_visible(False)
#         ax.grid(axis="x", color="#e0e0e0", linewidth=0.5)
#         ax.set_axisbelow(True)

#     for idx in range(len(PLOT_COLS), nrows * ncols):
#         axes[idx // ncols][idx % ncols].axis("off")

#     handles = [
#         plt.Line2D([0], [0], color=COLORS[l], linewidth=2, label=l)
#         for l in GLACIERS_TO_PLOT
#     ]
#     fig.legend(handles=handles,
#                loc="lower center",
#                ncol=3,
#                frameon=False,
#                fontsize=10,
#                bbox_to_anchor=(0.5, -0.02))
#     fig.suptitle(
#         f"Feature distributions — Aletsch vs Ålfotbreen vs {ala_label}",
#         fontsize=13)
#     plt.tight_layout()
#     plt.show()

# for display_name, df_gl in df_ala_by_glacier.items():
#     plot_glacier_vs_aletsch_aalfot(df_gl, f"{display_name} (ALA)")

### Maps:

In [ ]:
def plot_glacier_outline_map(
    glacier_outline_rgi,
    highlight_rgi_id=None,
    highlight_color="#2166ac",
    *,
    title="",
    figsize=(18, 10),
    extent=(5.8, 15, 44, 48),
    land_facecolor="lightgray",
    land_alpha=0.5,
    outline_color="steelblue",
    outline_alpha=0.5,
    add_features=True,
    add_gridlines=False,
    show=True,
):
    """
    Plot glacier outlines on a Cartopy map, optionally highlighting one glacier.

    Parameters
    ----------
    glacier_outline_rgi : GeoDataFrame
        RGI glacier outlines for the region.
    highlight_rgi_id : str or None
        RGIId of the glacier to highlight in highlight_color.
    highlight_color : str
        Color for the highlighted glacier.
    extent : tuple
        (lonW, lonE, latS, latN)
    """
    lonW, lonE, latS, latN = extent
    projPC = ccrs.PlateCarree()

    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=projPC)
    ax.set_extent([lonW, lonE, latS, latN], crs=ccrs.Geodetic())

    if add_features:
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.LAKES)
        ax.add_feature(cfeature.RIVERS)
        ax.add_feature(cfeature.BORDERS, linestyle="-", linewidth=1)
        ax.add_feature(cfeature.LAND,
                       facecolor=land_facecolor,
                       alpha=land_alpha)

    if add_gridlines:
        gl = ax.gridlines(draw_labels=True,
                          linewidth=0.3,
                          color="gray",
                          alpha=0.4,
                          linestyle="--")
        gl.top_labels = gl.right_labels = False
        gl.xlabel_style = {"size": 6}
        gl.ylabel_style = {"size": 6}

    # --- plot all outlines in region ---
    # clip to extent for speed
    gdf_clipped = glacier_outline_rgi.cx[lonW:lonE, latS:latN]
    if not gdf_clipped.empty:
        gdf_clipped.plot(ax=ax, transform=projPC, color="black", alpha=0.7)

    # --- highlight target glacier with a star marker ---
    if highlight_rgi_id is not None:
        gdf_target = glacier_outline_rgi[glacier_outline_rgi["RGIId"] ==
                                         highlight_rgi_id]
        if not gdf_target.empty:
            # get centroid of the target glacier
            centroid = gdf_target.geometry.centroid.iloc[0]
            ax.plot(
                centroid.x,
                centroid.y,
                marker="*",
                markersize=24,
                color=highlight_color,
                markeredgecolor="white",
                markeredgewidth=0.5,
                transform=projPC,
                zorder=4,
            )
        else:
            print(f"Warning: RGIId {highlight_rgi_id} not found in shapefile.")

    if title:
        ax.set_title(title,
                     fontsize=20,
                     fontweight="bold",
                     loc="left",
                     pad=3,
                     color=highlight_color)

    plt.tight_layout()
    if show:
        plt.show()

    return fig, ax

In [ ]:
GLACIERS_TO_PLOT = {
    "Aletsch (CH)": {
        "df": df_aletsch,
        "color": COLORS["Aletsch (CH)"],
        "rgi": "RGI60-11.01450"
    },
    "Ålfotbreen (NOR)": {
        "df": df_nor_by_glacier["Ålfotbreen"],
        "color": COLORS["Ålfotbreen (NOR)"],
        "rgi": "RGI60-08.02992"
    },
    "Kahiltna (ALA)": {
        "df": df_ala_by_glacier["Kahiltna"],
        "color": COLORS["Kahiltna (ALA)"],
        "rgi": "RGI60-01.00013"
    },
}

# load shapefiles once
gdf_ceu = gpd.read_file(
    os.path.join(cfg.dataPath,
                 "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp"))
gdf_nor = gpd.read_file(
    os.path.join(cfg.dataPath,
                 "RGI_v6/RGI_08_Scandinavia/08_rgi60_Scandinavia.shp"))
gdf_ala = gpd.read_file(
    os.path.join(cfg.dataPath, "RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp"))

# Aletsch
fig, ax = plot_glacier_outline_map(
    glacier_outline_rgi=gdf_ceu,
    highlight_rgi_id=GLACIERS_TO_PLOT["Aletsch (CH)"]["rgi"],
    highlight_color=GLACIERS_TO_PLOT["Aletsch (CH)"]["color"],
    title="Aletsch (CH)",
    extent=(5.8, 11, 45.5, 48),
)
# plt.savefig("figures/section1_parts/map_aletsch.pdf",
#             dpi=300,
#             bbox_inches="tight")
plt.close()

In [ ]:
# Ålfotbreen
fig, ax = plot_glacier_outline_map(
    glacier_outline_rgi=gdf_nor,
    highlight_rgi_id=GLACIERS_TO_PLOT["Ålfotbreen (NOR)"]["rgi"],
    highlight_color=GLACIERS_TO_PLOT["Ålfotbreen (NOR)"]["color"],
    title="Ålfotbreen (NOR)",
    extent=(4, 24, 58, 71),
)
# plt.savefig("figures/section1_parts/map_aalfotbreen.pdf",
#             dpi=300,
#             bbox_inches="tight")
plt.close()

In [ ]:
# Kahiltna
fig, ax = plot_glacier_outline_map(
    glacier_outline_rgi=gdf_ala,
    highlight_rgi_id=GLACIERS_TO_PLOT["Kahiltna (ALA)"]["rgi"],
    highlight_color=GLACIERS_TO_PLOT["Kahiltna (ALA)"]["color"],
    title="Kahiltna (ALA)",
    extent=(-170, -142, 55, 68),
)
# plt.savefig("figures/section1_parts/map_kahiltna.pdf",
#             dpi=300,
#             bbox_inches="tight")
plt.close()